# LC 435 — Non-overlapping Intervals
**Difficulty:** Medium | **Category:** Greedy | **Pattern:** Sort by End

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> To keep the <em>most</em> intervals,
always keep the one that ends earliest — it leaves the most room
for future intervals. Sort by end time, greedily keep non-overlapping
ones, and return n minus the count kept.
</div>

## Official Problem Statement

Given an array of `intervals` where `intervals[i] = [start_i, end_i]`,
return the **minimum number of intervals you need to remove** to make
the rest of the intervals non-overlapping.

**Note:** Intervals which only touch at a point (e.g. [1,2] and [2,3])
are **non-overlapping**.

**Constraints:**
- `1 <= intervals.length <= 10^5`
- `intervals[i].length == 2`
- `-5 * 10^4 <= start_i < end_i <= 5 * 10^4`

## What This Is Actually Asking

You have a set of intervals and some of them clash. You want to
remove as few as possible so that no two remaining intervals overlap.

The trick is: instead of asking "what to remove", ask "what is the
maximum number I can keep". Keep the most, remove the rest.

To keep the maximum non-overlapping set, always greedily keep the
interval that ends the soonest — it frees up the most future space.
This is the classic Activity Selection Problem.

## Walk Through an Example by Hand

`intervals = [[1,2],[2,3],[3,4],[1,3]]`

Step 1 — Sort by end:
```
[[1,2], [2,3], [1,3], [3,4]]
```

Step 2 — Sweep:
```
prev_end = 2, kept = 1   (take [1,2])

[2,3]: start=2 >= prev_end=2 → keep! prev_end=3, kept=2
[1,3]: start=1 <  prev_end=3 → SKIP (remove it)
[3,4]: start=3 >= prev_end=3 → keep! prev_end=4, kept=3
```

kept=3, n=4 → remove = 4 - 3 = **1**

Answer: `1` (remove [1,3])

## The Picture

```
Sorted by END:

[1,2]  ├─┤
[2,3]     ├─┤
[1,3]  ├───┤         ← overlaps with [2,3], ends later → REMOVE
[3,4]        ├─┤
       1  2  3  4

Why sort by END (not start)?
  Sorting by END ensures we always pick the interval
  that "ends earliest" first, maximizing leftover space.

Decision rule at each step:
  next.start >= prev_end  →  KEEP (no overlap)
  next.start <  prev_end  →  SKIP/REMOVE (overlaps)

Kept set: [1,2], [2,3], [3,4]  → 3 kept
Removed:  [1,3]                → 1 removed

Answer = n - max_kept = 4 - 3 = 1
```

## When To Use This Pattern

- When you see **"minimum removals to make non-overlapping"**, think
  *flip to: max intervals I can keep → sort by end*.
- When you see **"activity selection"** or **"max non-clashing jobs"**,
  think *sort by end time, greedy keep*.
- When two intervals **touch at a point** and are treated as valid,
  think *use `>=` not `>` in the keep condition*.
- When sorting by start fails your greedy intuition, think
  *try sorting by end instead*.
- When you need **min cost** to remove, not just count, think
  *same pattern but track cost instead of count*.

## The Approach

Sort the intervals by their end value. Initialize `prev_end` to the
end of the first interval and `kept` to 1. Iterate through the
remaining intervals: if the current interval's start is greater than
or equal to `prev_end`, it does not overlap — keep it, update
`prev_end`, and increment `kept`. Otherwise skip it (it overlaps and
ends later, so removing it is optimal). Return `n - kept`.

In [ ]:
from typing import List  # type hints for function signatures

In [ ]:
def test_harness(func):
    """Run all test cases against func and report results."""
    tests = [
        # (input_intervals, expected_removals)
        ([[1,2],[2,3],[3,4],[1,3]],      1),  # remove [1,3]
        ([[1,2],[1,2],[1,2]],            2),  # keep one, remove two
        ([[1,2],[2,3]],                  0),  # touch only, no overlap
        ([[1,100],[11,22],[1,11],[2,12]], 2),  # big interval eats others
        ([[1,2]],                        0),  # single interval
        (
            [[-52,31],[-73,-26],[82,97],
             [-65,-11],[-62,-49],[95,99],
             [58,95]],
            1
        ),                                    # negative values
        ([[0,2],[1,3],[2,4],[3,5],[4,6]], 2),  # chain with overlaps
    ]
    passed = 0
    for intervals, expected in tests:
        result = func([iv[:] for iv in intervals])  # pass copy
        status = "PASSED" if result == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | intervals={intervals}")
            print(f"        expected={expected}, got={result}")
        else:
            passed += 1
    total = len(tests)
    print(f"\n{passed}/{total} tests passed.")

In [ ]:
def erase_overlap_intervals(intervals: List[List[int]]) -> int:
    """
    Greedy: sort by end, keep max non-overlapping, return n - kept.

    Equivalent to Activity Selection Problem.
    Always greedily keep the interval with the earliest end —
    it leaves the most room for future intervals.

    Args:
        intervals: List of [start, end] pairs.
    Returns:
        Minimum number of intervals to remove.

    Time:  O(n log n)  — dominated by sort
    Space: O(1)        — in-place sort, counters only
    """
    pass


# --- Debug prints (remove before final submission) ---
iv1 = [[1,2],[2,3],[3,4],[1,3]]
print(erase_overlap_intervals(iv1))   # expected: 1

iv2 = [[1,2],[1,2],[1,2]]
print(erase_overlap_intervals(iv2))   # expected: 2

iv3 = [[1,2],[2,3]]
print(erase_overlap_intervals(iv3))   # expected: 0

iv4 = [[1,100],[11,22],[1,11],[2,12]]
print(erase_overlap_intervals(iv4))   # expected: 2

iv5 = [[0,2],[1,3],[2,4],[3,5],[4,6]]
print(erase_overlap_intervals(iv5))   # expected: 2

In [ ]:
# Uncomment and run when solution is ready
# test_harness(erase_overlap_intervals)

## Complexity

| Approach                        | Time       | Space |
|---------------------------------|------------|-------|
| Brute Force (try all subsets)   | O(2^n)     | O(n)  |
| DP (interval scheduling)        | O(n²)      | O(n)  |
| Greedy Sort by End              | O(n log n) | O(1)  |

## Real World Connection

At Citi, trade reconciliation jobs compete for the same database
connection windows. This problem models how many jobs to cancel
(minimum removals) so the rest can run without locking conflicts.

On AWS, Lambda concurrency slots are finite. When burst traffic
causes too many overlapping execution windows, this greedy algorithm
determines the minimum number of invocations to throttle so the
remaining ones complete without resource contention.

In data engineering, when multiple Glue jobs write to the same
partition window, you must cancel the minimum number of jobs.
Sorting by end time and greedily keeping non-overlapping jobs is
the optimal scheduling strategy for maximizing pipeline throughput.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra